# 資料處理

## 檢查版本

In [3]:
import sys, xarray as xr
print("Python exe:", sys.executable)
print("Engines:", xr.backends.list_engines())

Python exe: /home/jundian/miniconda3/envs/geospatial-neural-adapter/bin/python
Engines: {'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'h5netcdf': <H5netcdfBackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


## 讀nc4

In [4]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:13<00:00, 41.24it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

In [5]:
# ===== 只看美國本土範圍，並從中抽樣 50 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 50 個（如果不足 50，就全部使用）
n_sample = min(50, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 50 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


## VARIMA (跑超久........)

In [6]:
# # ===== VARIMA on the 50 sampled US grid cells =====
# import numpy as np
# import pandas as pd
# from typing import Tuple
# from darts import TimeSeries
# from darts.models import VARIMA
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# print("\n===== VARIMA baseline on 50 sampled US cells =====")

# # 1. 建 target DataFrame：50 個格點 × T 時間點（原始單位）
# SAMPLE_SIZE = y_sample.shape[0]   # 預期是 50
# T_total = y_sample.shape[1]

# ts_df = pd.DataFrame(
#     y_sample.T,  # (T, 50)
#     index=time_index,
#     columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
# ).astype("float32")

# print("Total time steps:", T_total)
# print("ts_df shape:", ts_df.shape)  # (T, 50)

# # 2. 切成 70 / 15 / 15（train / val / test）
# train_frac = 0.7
# val_frac   = 0.15   # test 也是 0.15

# cut_train = int(T_total * train_frac)
# cut_val   = int(T_total * (train_frac + val_frac))

# idx_time = ts_df.index
# split_1_time = idx_time[cut_train]
# split_2_time = idx_time[cut_val]

# ts_all_raw = TimeSeries.from_dataframe(ts_df)

# train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
# val_raw,   test_raw = tmp_raw.split_before(split_2_time)

# print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# # 3. 用 train 統計量標準化
# train_df = ts_df.iloc[:cut_train]

# mean_vec = train_df.mean(axis=0)
# std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

# ts_df_scaled = (ts_df - mean_vec) / std_vec
# ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

# train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
# val_ts,   test_ts = tmp_ts.split_before(split_2_time)

# T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
# print("\n[Scaled TimeSeries lengths]")
# print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# # 真實值（原單位，用來算 MSE/MAE/R²）
# vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T, 50)
# val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val, 50)
# test_true_raw = vals_all[cut_val:, :]             # (T_test, 50)

# def inverse_scale(x: np.ndarray) -> np.ndarray:
#     """
#     x: shape (T_horizon, C)
#     回傳反標準化後的值，shape 相同
#     """
#     return x * std_vec.values + mean_vec.values

# # ======================================================
# # 3.5 在 validation 上做 VARIMA 階數選擇 (order selection)
# # ======================================================
# def eval_varima_order(order: Tuple[int, int, int]) -> float:
#     """
#     給定 (p, d, q)，在 train_ts 上 fit，
#     在 val_ts 上預測，回傳 val RMSE（原單位、50 cell flatten 後）。
#     """
#     try:
#         print(f"  Try order={order} ...")
#         model = VARIMA(*order)   # 相當於 VARIMA(p, d, q)
#         model.fit(train_ts)

#         pred_val = model.predict(n=T_val)
#         pred_vals = np.asarray(pred_val.all_values(copy=False))
#         if pred_vals.ndim == 3:
#             pred_vals = pred_vals[..., 0]  # (T_val, C)

#         # 反標準化回原單位
#         pred_val_raw = inverse_scale(pred_vals)  # (T_val, 50)

#         y_true = val_true_raw.reshape(-1, SAMPLE_SIZE)
#         y_pred = pred_val_raw.reshape(-1, SAMPLE_SIZE)

#         rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#         print(f"    VAL RMSE = {rmse:.4f}")
#         return rmse
#     except Exception as e:
#         print(f"    order={order} failed with error: {e}")
#         return np.inf

# # 你可以自己改這裡的搜尋範圍：
# # 目前：只搜尋 (p, d, q) = (1~5, 0, 0) 當作 VAR(p)
# candidate_p = [1, 2, 3, 4, 5]
# candidate_d = [0]
# candidate_q = [0]

# best_order = None
# best_rmse = np.inf

# print("\n[Order selection on VAL]")
# for p in candidate_p:
#     for d in candidate_d:
#         for q in candidate_q:
#             order = (p, d, q)
#             rmse = eval_varima_order(order)
#             if rmse < best_rmse:
#                 best_rmse = rmse
#                 best_order = order

# print(f"\nBest order by VAL RMSE: {best_order}, RMSE = {best_rmse:.4f}")

# # 用選到的 best_order 當作正式的 ORDER
# ORDER = best_order

# # ======================================================
# # 4. VARIMA：train → predict val（用選出的 ORDER）
# # ======================================================
# print(f"\n[Step] VARIMA: train → predict VAL with ORDER={ORDER}")

# model_varima_val = VARIMA(*ORDER)
# model_varima_val.fit(train_ts)

# pred_var_val = model_varima_val.predict(n=T_val)   # 預測 val 長度
# pred_var_val_vals = np.asarray(pred_var_val.all_values(copy=False))
# # shape 可能是 (T_val, C, 1)，把最後一維壓掉
# if pred_var_val_vals.ndim == 3:
#     pred_var_val_vals = pred_var_val_vals[..., 0]

# pred_var_val_raw = inverse_scale(pred_var_val_vals)   # (T_val, 50)

# # 5. VARIMA：train+val → predict test
# print(f"[Step] VARIMA: train+val → predict TEST with ORDER={ORDER}")

# train_val_ts = train_ts.concatenate(val_ts)

# model_varima_test = VARIMA(*ORDER)
# model_varima_test.fit(train_val_ts)

# pred_var_test = model_varima_test.predict(n=T_test)
# pred_var_test_vals = np.asarray(pred_var_test.all_values(copy=False))
# if pred_var_test_vals.ndim == 3:
#     pred_var_test_vals = pred_var_test_vals[..., 0]

# pred_var_test_raw = inverse_scale(pred_var_test_vals)  # (T_test, 50)

# # 6. 計算 VAL / TEST 的 MSE / MAE / RMSE / R²
# y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
# y_val_pred  = pred_var_val_raw.reshape(-1, SAMPLE_SIZE)
# y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
# y_test_pred = pred_var_test_raw.reshape(-1, SAMPLE_SIZE)

# mse_var_val  = mean_squared_error(y_val_true,  y_val_pred)
# mae_var_val  = mean_absolute_error(y_val_true, y_val_pred)
# mse_var_test = mean_squared_error(y_test_true, y_test_pred)
# mae_var_test = mean_absolute_error(y_test_true, y_test_pred)

# rmse_var_val  = np.sqrt(mse_var_val)
# rmse_var_test = np.sqrt(mse_var_test)

# r2_var_val  = r2_score(y_val_true,  y_val_pred)
# r2_var_test = r2_score(y_test_true, y_test_pred)

# print("\n===== VARIMA RESULTS (50 sampled US cells) =====")
# print(f"VAL  RMSE: {rmse_var_val:.4f}, MSE: {mse_var_val:.4f}, MAE: {mae_var_val:.4f}, R²: {r2_var_val:.4f}")
# print(f"TEST RMSE: {rmse_var_test:.4f}, MSE: {mse_var_test:.4f}, MAE: {mae_var_test:.4f}, R²: {r2_var_test:.4f}")

# metrics_table_varima = pd.DataFrame({
#     "Model": ["VARIMA", "VARIMA"],
#     "Split": ["VAL",     "TEST"],
#     "RMSE":  [rmse_var_val,  rmse_var_test],
#     "MSE":   [mse_var_val,   mse_var_test],
#     "MAE":   [mae_var_val,   mae_var_test],
#     "R2":    [r2_var_val,    r2_var_test],
# })

# print("\nMSE / MAE / RMSE / R² table (VARIMA, 50 cells):")
# print(metrics_table_varima.to_string(index=False))


## VAR (Version2)

In [7]:
# ===== VARIMA on the 50 sampled US grid cells =====
import numpy as np
import pandas as pd
from typing import Tuple
from darts import TimeSeries
from darts.models import VARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VARIMA baseline on 50 sampled US grid cells =====")

# 1. 建 target DataFrame：50 個格點 × T 時間點（原始單位）
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 50
T_total = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,  # (T, 50)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T_total)
print("ts_df shape:", ts_df.shape)  # (T, 50)

# 2. 切成 70 / 15 / 15（train / val / test）
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

idx_time = ts_df.index
split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)

train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 3. 用 train 統計量標準化
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T, 50)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val, 50)
test_true_raw = vals_all[cut_val:, :]             # (T_test, 50)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳反標準化後的值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中用粗體標出每個指標的最佳（最小）值 ----
# 找出各指標的最佳 index（用數值的 ic_table）
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

# 建一個純字串版的 printed_table，避免 dtype 警告
printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"]],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"]],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"]],
})

# 在最佳位置加粗體標記（markdown 風格）
printed_table.loc[best_idx_aic,  "AIC"]  = printed_table.loc[best_idx_aic,  "AIC"]  + "*"
printed_table.loc[best_idx_bic,  "BIC"]  = printed_table.loc[best_idx_bic,  "BIC"]  + "*"
printed_table.loc[best_idx_hqic, "HQIC"] = printed_table.loc[best_idx_hqic, "HQIC"] + "*"
printed_table.loc[best_idx_fpe,  "FPE"]  = printed_table.loc[best_idx_fpe,  "FPE"]  + "*"

print("\nInformation Criteria table (p = 1..{}):".format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 用 BIC 選 lag（這裡還是用數值 ic_table）
p_selected = int(ic_table.loc[best_idx_bic, "lag"])
print(f"\nSelected lag by BIC = {p_selected}")

ORDER = (p_selected, 0, 0)
print(f"Final VARIMA ORDER = {ORDER}")

# ======================================================
# 4. VARIMA：train → predict val（用選出的 ORDER）
# ======================================================
print(f"\n[Step] VARIMA: train → predict VAL with ORDER={ORDER}")

model_varima_val = VARIMA(*ORDER)
model_varima_val.fit(train_ts)

pred_var_val = model_varima_val.predict(n=T_val)   # 預測 val 長度
pred_var_val_vals = np.asarray(pred_var_val.all_values(copy=False))
# shape 可能是 (T_val, C, 1)，把最後一維壓掉
if pred_var_val_vals.ndim == 3:
    pred_var_val_vals = pred_var_val_vals[..., 0]

pred_var_val_raw = inverse_scale(pred_var_val_vals)   # (T_val, 50)

# ======================================================
# 5. VARIMA：train+val → predict test（同一個 ORDER）
# ======================================================
print(f"[Step] VARIMA: train+val → predict TEST with ORDER={ORDER}")

train_val_ts = train_ts.concatenate(val_ts)

model_varima_test = VARIMA(*ORDER)
model_varima_test.fit(train_val_ts)

pred_var_test = model_varima_test.predict(n=T_test)
pred_var_test_vals = np.asarray(pred_var_test.all_values(copy=False))
if pred_var_test_vals.ndim == 3:
    pred_var_test_vals = pred_var_test_vals[..., 0]

pred_var_test_raw = inverse_scale(pred_var_test_vals)  # (T_test, 50)

# ======================================================
# 6. 計算 VAL / TEST 的 MSE / MAE / RMSE / R²
# ======================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_var_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_var_test_raw.reshape(-1, SAMPLE_SIZE)

mse_var_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_var_val  = mean_absolute_error(y_val_true, y_val_pred)
mse_var_test = mean_squared_error(y_test_true, y_test_pred)
mae_var_test = mean_absolute_error(y_test_true, y_test_pred)

rmse_var_val  = np.sqrt(mse_var_val)
rmse_var_test = np.sqrt(mse_var_test)

r2_var_val  = r2_score(y_val_true,  y_val_pred)
r2_var_test = r2_score(y_test_true, y_test_pred)

print("\n===== VARIMA RESULTS (50 sampled US cells) =====")
print(f"VAL  RMSE: {rmse_var_val:.4f}, MSE: {mse_var_val:.4f}, MAE: {mae_var_val:.4f}, R²: {r2_var_val:.4f}")
print(f"TEST RMSE: {rmse_var_test:.4f}, MSE: {mse_var_test:.4f}, MAE: {mae_var_test:.4f}, R²: {r2_var_test:.4f}")

metrics_table_varima = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA"],
    "Split": ["VAL",     "TEST"],
    "RMSE":  [rmse_var_val,  rmse_var_test],
    "MSE":   [mse_var_val,   mse_var_test],
    "MAE":   [mae_var_val,   mae_var_test],
    "R2":    [r2_var_val,    r2_var_test],
})

print("\nMSE / MAE / RMSE / R² table (VARIMA, 50 cells):")
print(metrics_table_varima.to_string(index=False))



===== VARIMA baseline on 50 sampled US grid cells =====
Total time steps: 548
ts_df shape: (548, 50)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled TimeSeries lengths]
T_train = 383 T_val = 82 T_test = 83

[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-244.1342, BIC=-217.7970, HQIC=-233.6856, FPE=1.0202e-106
  p=2 OK: AIC=-241.5810, BIC=-189.3208, HQIC=-220.8460, FPE=2.3140e-105
  p=3 OK: AIC=-239.8699, BIC=-161.5849, HQIC=-208.8061, FPE=6.7809e-104
  p=4 OK: AIC=-241.4986, BIC=-137.0863, HQIC=-200.0632, FPE=5.4500e-103
  p=5 OK: AIC=-249.1335, BIC=-118.4908, HQIC=-197.2834, FPE=5.0931e-103
  p=6 OK: AIC=-274.5520, BIC=-117.5751, HQIC=-212.2439, FPE=4.0735e-107
  p=7 估計失敗（28-th leading minor of the array is not positive definite），停止往更大 p 試。

Information Criteria table (p = 1..6):
lag        AIC        BIC       HQIC          FPE
  1  -244.1342 -217.7970* -233.6856*  1.0202e-106
  2  -241.5810  -189.3208  -220.8460  2.3140e-105
  

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



===== VARIMA RESULTS (50 sampled US cells) =====
VAL  RMSE: 3.8842, MSE: 15.0869, MAE: 2.8543, R²: 0.7072
TEST RMSE: 3.9091, MSE: 15.2812, MAE: 2.8954, R²: 0.7056

MSE / MAE / RMSE / R² table (VARIMA, 50 cells):
 Model Split     RMSE       MSE      MAE       R2
VARIMA   VAL 3.884192 15.086945 2.854295 0.707181
VARIMA  TEST 3.909113 15.281168 2.895363 0.705635


### VARIMA Results — 50 Sampled US Grid Cells

**Data Summary**  
- Total time steps: **548**  
- Grid cells: **50**  
- Train length: **383**  
- Validation length: **82**  
- Test length: **83**

---

### Selected Order (Manual VAR(p) Grid Search)

| lag |    AIC    |     BIC     |    HQIC    |      FPE       |
|----:|----------:|------------:|-----------:|---------------:|
| 1   | -244.1342 | **-217.7970** | **-233.6856** | 1.0202e-106 |
| 2   | -241.5810 | -189.3208 | -220.8460 | 2.3140e-105 |
| 3   | -239.8699 | -161.5849 | -208.8061 | 6.7809e-104 |
| 4   | -241.4986 | -137.0863 | -200.0632 | 5.4500e-103 |
| 5   | -249.1335 | -118.4908 | -197.2834 | 5.0931e-103 |
| 6   | **-274.5520** | -117.5751 | -212.2439 | **4.0735e-107** |

- Selected lag by **BIC**: **1**  
- Final VARIMA order: **(1, 0, 0)**

---

### Performance Summary

| Model  | Split |   RMSE   |    MSE    |   MAE   |    R²    |
|:------:|:-----:|:--------:|:---------:|:-------:|:--------:|
| VARIMA |  VAL  | 3.884192 | 15.086945 | 2.854295 | 0.707181 |
| VARIMA | TEST  | 3.909113 | 15.281168 | 2.895363 | 0.705635 |


### Residual ACF/PACF

In [8]:
# ===== VARIMA Residual ACF / PACF for 50 sampled US cells =====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 這些變數需已在前一個 VARIMA cell 中建立：
# val_true_raw      : (T_val,  50)
# test_true_raw     : (T_test, 50)
# pred_var_val_raw  : (T_val,  50)
# pred_var_test_raw : (T_test, 50)
# ts_df             : (T_total, 50)，index = time_index
# cut_train, cut_val

output_dir = "photo/VARIMA_resid_acf_pacf_50cells"
os.makedirs(output_dir, exist_ok=True)
print(f"Residual ACF/PACF will be saved to: {output_dir}")

# ===== 1. 組出 residual（原單位） =====
resid_val  = val_true_raw  - pred_var_val_raw    # (T_val,  50)
resid_test = test_true_raw - pred_var_test_raw   # (T_test, 50)

# 接成一條：val 接 test
resid_all = np.vstack([resid_val, resid_test])   # (T_val + T_test, 50)

# 對應時間 index（val + test）
idx_val  = ts_df.index[cut_train:cut_val]
idx_test = ts_df.index[cut_val:]
idx_all  = pd.Index(idx_val.tolist() + idx_test.tolist())

T_resid, C = resid_all.shape
names = ts_df.columns

print(f"VARIMA residuals: T={T_resid}, C={C}")

MAX_LAG = 40   # ACF / PACF 最大落後期，可依需要調整

# ===== 2. 逐 cell 畫 ACF / PACF =====
for j in range(C):
    name = str(names[j])

    series_resid = pd.Series(resid_all[:, j], index=idx_all).dropna()

    if len(series_resid) <= MAX_LAG + 5:
        print(f"Skip {name}: residual series too short (len={len(series_resid)})")
        continue

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

    # ACF
    plot_acf(series_resid, lags=MAX_LAG, ax=ax1)
    ax1.set_title(f"VARIMA Residual ACF - {name}")

    # PACF
    plot_pacf(series_resid, lags=MAX_LAG, ax=ax2, method="ywm")
    ax2.set_title(f"VARIMA Residual PACF - {name}")
    ax2.set_xlabel("Lag")

    plt.tight_layout()

    safe_name = name.replace(" ", "_").replace("/", "_")
    save_path = os.path.join(output_dir, f"{safe_name}_varima_resid_acf_pacf.png")

    plt.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"Saved: {save_path}")

print("Done: all VARIMA residual ACF/PACF plots saved.")


Residual ACF/PACF will be saved to: photo/VARIMA_resid_acf_pacf_50cells
VARIMA residuals: T=165, C=50
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_0_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_1_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_2_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_3_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_4_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_5_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_6_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_7_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_8_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_9_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_10_varima_resid_acf_pacf.png
Saved: photo/VARIMA_resid_acf_pacf_50cells/cell_11_varima_res

#### Fixed CI

In [9]:
# ===== VARIMA Residual ACF / PACF for all cells (fixed 95% CI) =====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 這些變數需已在前一個 VARIMA cell 中建立：
# val_true_raw      : (T_val,  C)
# test_true_raw     : (T_test, C)
# pred_var_val_raw  : (T_val,  C)
# pred_var_test_raw : (T_test, C)
# ts_df             : (T_total, C)，index = time_index
# cut_train, cut_val

output_dir = "photo/VARIMA_resid_acf_pacf_50cells_fixed_CI"
os.makedirs(output_dir, exist_ok=True)
print(f"Residual ACF/PACF will be saved to: {output_dir}")

# ===== 1. 組出 residual（原單位） =====
resid_val  = val_true_raw  - pred_var_val_raw    # (T_val,  C)
resid_test = test_true_raw - pred_var_test_raw   # (T_test, C)

# 接成一條：val 接 test
resid_all = np.vstack([resid_val, resid_test])   # (T_val + T_test, C)

# 對應時間 index（val + test）
idx_val  = ts_df.index[cut_train:cut_val]
idx_test = ts_df.index[cut_val:]
idx_all  = pd.Index(idx_val.tolist() + idx_test.tolist())

T_resid, C = resid_all.shape
names = ts_df.columns

print(f"VARIMA residuals: T={T_resid}, C={C}")

MAX_LAG = 40   # ACF / PACF 最大落後期，可依需要調整

# ===== 2. 逐 cell 畫 ACF / PACF =====
for j in range(C):
    name = str(names[j])

    series_resid = pd.Series(resid_all[:, j], index=idx_all).dropna()
    n = len(series_resid)

    if n <= MAX_LAG + 5:
        print(f"Skip {name}: residual series too short (len={n})")
        continue

    print(f"Plotting residual ACF/PACF for {name}, length = {n}")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

    # 1) ACF / PACF：關掉內建 CI（alpha=None）
    plot_acf(series_resid, lags=MAX_LAG, ax=ax1, alpha=None)
    ax1.set_title(f"VARIMA Residual ACF - {name}")

    plot_pacf(series_resid, lags=MAX_LAG, ax=ax2, method="ywm", alpha=None)
    ax2.set_title(f"VARIMA Residual PACF - {name}")
    ax2.set_xlabel("Lag")

    # 2) 自訂淺藍色 95% CI：±1.96 / sqrt(n)
    conf = 1.96 / np.sqrt(n)
    for ax in (ax1, ax2):
        ax.axhline(0.0,  linestyle="-",  linewidth=1,   color="black")
        ax.axhline(conf,  linestyle="--", linewidth=1.2, color="blue")
        ax.axhline(-conf, linestyle="--", linewidth=1.2, color="blue")

    plt.tight_layout()

    safe_name = name.replace(" ", "_").replace("/", "_")
    save_path = os.path.join(output_dir, f"{safe_name}_varima_resid_acf_pacf_fixed_CI.png")

    plt.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"Saved: {save_path}")

print("Done: all VARIMA residual ACF/PACF plots with fixed CI saved.")


Residual ACF/PACF will be saved to: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI
VARIMA residuals: T=165, C=50
Plotting residual ACF/PACF for cell_0, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_0_varima_resid_acf_pacf_fixed_CI.png
Plotting residual ACF/PACF for cell_1, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_1_varima_resid_acf_pacf_fixed_CI.png
Plotting residual ACF/PACF for cell_2, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_2_varima_resid_acf_pacf_fixed_CI.png
Plotting residual ACF/PACF for cell_3, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_3_varima_resid_acf_pacf_fixed_CI.png
Plotting residual ACF/PACF for cell_4, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_4_varima_resid_acf_pacf_fixed_CI.png
Plotting residual ACF/PACF for cell_5, length = 165
Saved: photo/VARIMA_resid_acf_pacf_50cells_fixed_CI/cell_5_varima_resid_acf_pacf_fixed_CI.png
Plotting resi

### Raw data & mean & predict

In [10]:
# ===== VARIMA Raw data + Mean + Forecast (VAL/TEST) =====
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

out_dir = "photo/VARIMA_raw_mean_forecast_50cells"
os.makedirs(out_dir, exist_ok=True)
print(f"Figures will be saved to: {out_dir}")

# ---------- 1. 原始資料 (T_total, 50) ----------
vals = ts_df.to_numpy(dtype=np.float32)   # (T_total, 50)
idx_full = ts_df.index
C = vals.shape[1]
names = ts_df.columns

# ---------- 2. 取出 val / test 的時間軸 ----------
val_idx  = val_ts.time_index
test_idx = test_ts.time_index

T_val  = len(val_idx)
T_test = len(test_idx)

# ---------- 3. 從「val 的前一年」開始畫 ----------
start_plot = split_1_time - pd.DateOffset(years=1)
if start_plot < idx_full[0]:
    start_plot = idx_full[0]

# 原始資料：從 start_plot 之後
mask_raw = idx_full >= start_plot
idx_raw  = idx_full[mask_raw]
vals_raw = vals[mask_raw, :]   # (T_raw, 50)

# VAL 預測：時間 >= start_plot 的部分（用 VARIMA 的預測）
mask_val_use   = val_idx >= start_plot
idx_val_use    = val_idx[mask_val_use]
pred_val_use   = pred_var_val_raw[mask_val_use, :]   # (T_val_use, 50)

# TEST 預測：時間 >= start_plot 的部分（用 VARIMA 的預測）
mask_test_use  = test_idx >= start_plot
idx_test_use   = test_idx[mask_test_use]
pred_test_use  = pred_var_test_raw[mask_test_use, :]  # (T_test_use, 50)

# 灰底開始 / 結束時間（預測區間整體）
shade_start = split_1_time
shade_end   = idx_raw[-1]

print(f"raw length from start_plot = {len(idx_raw)}")
print(f"val pred length from start_plot = {len(idx_val_use)}")
print(f"test pred length from start_plot = {len(idx_test_use)}")
print(f"start_plot = {start_plot}, split_1_time = {split_1_time}")

# ---------- 4. 逐 cell 畫圖並存檔 ----------
for j in range(C):
    # 該 cell 從 start_plot 之後的 raw
    y_j = vals_raw[:, j]
    mean_j = float(np.mean(y_j))

    # 該 cell VAL / TEST 的 VARIMA 預測
    yhat_val_j  = pred_val_use[:, j]
    yhat_test_j = pred_test_use[:, j]

    plt.figure(figsize=(12, 5))

    # 1) raw data (start_plot 之後) —— 黑線
    plt.plot(idx_raw, y_j,
             color="black", linewidth=1.3, label="Raw data")

    # 2) 這一段時間的平均 —— 灰色虛線
    plt.axhline(mean_j,
                color="grey", linestyle="--", linewidth=1.2,
                label="Mean")

    # 3) 預測區間灰底（train → val 之後）
    plt.axvspan(shade_start, shade_end,
                color="grey", alpha=0.15, label="Forecast horizon")

    # 4) VARIMA VAL 預測 —— 藍色虛線
    if len(idx_val_use) > 0:
        plt.plot(idx_val_use, yhat_val_j,
                 color="blue", linestyle="--", linewidth=1.5,
                 label="VARIMA forecast (VAL)")

    # 5) VARIMA TEST 預測 —— 紅色虛線
    if len(idx_test_use) > 0:
        plt.plot(idx_test_use, yhat_test_j,
                 color="red", linestyle="--", linewidth=1.5,
                 label="VARIMA forecast (TEST)")

    # 6) 預測起點垂直線（train → val 的分割點）—— 灰色點線
    if split_1_time >= idx_raw[0]:
        plt.axvline(split_1_time, color="grey", linestyle=":", linewidth=1)

    # 如果你有 sample_idx_global，就跟原本一樣留著，方便 debug
    try:
        global_idx = int(sample_idx_global[j])
    except NameError:
        global_idx = None

    # 標題格式跟 SARIMA 一樣，只把字改成 VARIMA
    plt.title(f"cell_{j}: Raw, mean & VARIMA forecast (VAL/TEST)")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.grid(alpha=0.3)

    # legend 放在圖外上方（跟 SARIMA 一樣）
    plt.legend(
        loc="upper left",
        bbox_to_anchor=(0.01, 1.02),
        ncol=4,
        framealpha=0.9,
        fontsize=9
    )

    plt.tight_layout()

    fname = f"cell_{j:02d}_val_test_VARIMA.png"
    save_path = os.path.join(out_dir, fname)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"[{j+1}/{C}] Saved: {save_path}")

print(f"已將 {C} 張『VARIMA raw/mean + VAL/TEST forecast』圖存到資料夾：{out_dir}")


Figures will be saved to: photo/VARIMA_raw_mean_forecast_50cells
raw length from start_plot = 177
val pred length from start_plot = 82
test pred length from start_plot = 83
start_plot = 2010-12-01 00:30:00, split_1_time = 2011-12-01 00:30:00
[1/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_00_val_test_VARIMA.png
[2/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_01_val_test_VARIMA.png
[3/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_02_val_test_VARIMA.png
[4/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_03_val_test_VARIMA.png
[5/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_04_val_test_VARIMA.png
[6/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_05_val_test_VARIMA.png
[7/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_06_val_test_VARIMA.png
[8/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_07_val_test_VARIMA.png
[9/50] Saved: photo/VARIMA_raw_mean_forecast_50cells/cell_08_val_test_VARIMA.png
[10/50] Saved: photo/VARIMA_r

### Raw data & mean & predict mean

In [11]:
# ===== VARIMA Raw data + Raw mean + Forecast mean (VAL/TEST) =====
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 需要已經有：
# - ts_df               : (T_total, 50)，index = time_index
# - pred_var_val_raw    : (T_val,  50)
# - pred_var_test_raw   : (T_test, 50)
# - val_ts, test_ts     : Darts TimeSeries（用來拿時間軸）
# - split_1_time        : train/val 分割點（第一個 val 的時間）

out_dir = "photo"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "raw_mean_and_VARIMA_pred_mean.png")
print(f"Figure will be saved to: {out_path}")

# ---------- 1. 原始資料 (T_total, 50) ----------
vals = ts_df.to_numpy(dtype=np.float32)   # (T_total, 50)
idx_full = ts_df.index
C = vals.shape[1]

# ---------- 2. val / test 時間軸 ----------
val_idx  = val_ts.time_index
test_idx = test_ts.time_index

# ---------- 3. 從「val 的前一年」開始畫 ----------
start_plot = split_1_time - pd.DateOffset(years=1)
if start_plot < idx_full[0]:
    start_plot = idx_full[0]

# 原始資料：從 start_plot 之後
mask_raw = idx_full >= start_plot
idx_raw  = idx_full[mask_raw]
vals_raw = vals[mask_raw, :]          # (T_raw, 50)
raw_mean = vals_raw.mean(axis=1)      # (T_raw,)

# VAL / TEST 預測時間與 mean（VARIMA）
pred_val_mean  = pred_var_val_raw.mean(axis=1)   # (T_val,)
pred_test_mean = pred_var_test_raw.mean(axis=1)  # (T_test,)

mask_val_use       = val_idx >= start_plot
idx_val_use        = val_idx[mask_val_use]
pred_val_mean_use  = pred_val_mean[mask_val_use]

mask_test_use      = test_idx >= start_plot
idx_test_use       = test_idx[mask_test_use]
pred_test_mean_use = pred_test_mean[mask_test_use]

# 灰底開始 / 結束時間（預測區間）
shade_start = split_1_time
shade_end   = idx_raw[-1]

print(f"raw length from start_plot = {len(idx_raw)}")
print(f"val pred mean length from start_plot = {len(idx_val_use)}")
print(f"test pred mean length from start_plot = {len(idx_test_use)}")
print(f"start_plot = {start_plot}, split_1_time = {split_1_time}")

# ---------- 4. 一張圖：all raw + raw mean + VARIMA 預測 mean ----------
plt.figure(figsize=(12, 5))

# (a) 所有 50 條 raw（透明）
for j in range(C):
    plt.plot(idx_raw, vals_raw[:, j],
             linewidth=0.5,
             alpha=0.4)

# (b) raw mean（黑色粗線）
plt.plot(idx_raw, raw_mean,
         color="black", linewidth=2.0, label="Raw mean")

# (c) VAL / TEST 預測 mean（VARIMA）
if len(idx_val_use) > 0:
    plt.plot(idx_val_use, pred_val_mean_use,
             color="blue", linestyle="--", linewidth=2.0,
             label="VARIMA forecast mean (VAL)")

if len(idx_test_use) > 0:
    plt.plot(idx_test_use, pred_test_mean_use,
             color="red", linestyle="--", linewidth=2.0,
             label="VARIMA forecast mean (TEST)")

# (d) 預測區間灰底
plt.axvspan(shade_start, shade_end,
            color="grey", alpha=0.15, label="Forecast horizon")

# (e) train → val 分割線
if split_1_time >= idx_raw[0]:
    plt.axvline(split_1_time, color="grey", linestyle=":", linewidth=1)

plt.title("Raw data, raw mean & VARIMA forecast mean (50 sampled US grid cells)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.grid(alpha=0.3)

# legend 放在圖外上方
plt.legend(
    loc="upper left",
    bbox_to_anchor=(0.01, 1.02),
    ncol=4,
    framealpha=0.9,
    fontsize=9
)

plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()

print(f"Figure saved to: {out_path}")


Figure will be saved to: photo/raw_mean_and_VARIMA_pred_mean.png
raw length from start_plot = 177
val pred mean length from start_plot = 82
test pred mean length from start_plot = 83
start_plot = 2010-12-01 00:30:00, split_1_time = 2011-12-01 00:30:00
Figure saved to: photo/raw_mean_and_VARIMA_pred_mean.png
